![image_1781088390501.png](./image_1781088390501.png "image_1781088390501.png")

![image_1781088361143.png](./image_1781088361143.png "image_1781088361143.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("OrdersDF").getOrCreate()

# Dataset
data = [
    (1, "2024-01-05", 1200),
    (2, "2024-01-18", 800),
    (3, "2024-01-29", 500),
    (4, "2024-02-03", 2100),
    (5, "2024-02-14", 900),
    (6, "2024-03-07", 1800),
    (7, "2024-03-22", 1200),
    (8, "2024-04-10", 4500),
    (9, "2024-04-25", 500)
]

# Define schema (column names)
columns = ["id", "order_date", "amount"]

# Create DataFrame
orders_df = spark.createDataFrame(data, columns)

# Show DataFrame
orders_df.show()


In [0]:
result_df = (
    orders_df.withColumn("year_month", f.date_format("order_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(f.sum("amount").alias("monthly_revenue"))
    .withColumn(
        "prev_month_revenue",
        f.lag("monthly_revenue").over(Window.orderBy("year_month")),
    )
    .withColumn(
        "growth_rate",
        f.round(
            (f.col("monthly_revenue") - f.col("prev_month_revenue"))
            * 100
            / f.col("prev_month_revenue"),
            1,
        ),
    )
    .orderBy("year_month")
)
display(result_df)